# ST-GCN — qual combinação de features vence?

A PoC anterior mostrou duas coisas que mudam o alvo do projeto:

| | ST-GCN | ResNet-18 |
|---|---|---|
| controle (x,y) | 72,1% | 93,0% |
| **+ ossos** | **91,0%** (+18,9, **8 de 8 folds**) | — |
| + z | +8,4 (7 de 8 folds) | −2,9 (**0** de 8) |
| parâmetros | **0,46M** | 11,25M |

O GCN com ossos chegou a 2 pontos da ResNet com **24× menos parâmetros**. Este notebook
testa o que mais cabe em cima disso.

## O que roda, e por quê

Tudo parte de `ossos`, que é o que já ganhou. Cada variante muda **uma coisa**.

| Variante | Muda | Custo em parâmetros |
|---|---|---|
| `G-ossos` | — (**controle desta sessão**) | 464.510 |
| `G-ossos-z` | + profundidade (ossos 3D) | +612 |
| `G-ossos-mov` | + velocidade das juntas e dos ossos | +1.224 |
| `G-ossos-z-mov` | os dois | +2.448 |
| `G-ossos-adapt` | adjacência aprendida (mão↔rosto) | +6.498 |
| `G-ossos-k5` | kernel temporal 9→5 | **−155.520** |

`k5` é o único que **encolhe**: ~80% dos parâmetros do modelo estão nas convoluções
temporais, e esse kernel nunca foi calibrado.

## Por que não uma grade completa

Com 8 folds e ruído de ~1,7 pp, rodar muitas variantes e **pegar o máximo infla o
número** — o vencedor ganha em parte por sorte. Por isso a última etapa é automática:
o notebook reroda o melhor com **outra semente**. Se o ganho sobreviver, é real.

## Orçamento de tempo

Cada execução do GCN é ~55 min (medido na sessão anterior). O notebook tem um **guarda de
tempo**: antes de cada variante ele confere se cabe no limite e, se não couber, pula e
relata em vez de ser morto no meio.


## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

# ORDEM IMPORTA, e custou 68 min de cota descobrir: o editor novo do Kaggle é
# baseado em Colab, então `google.colab` está no sys.modules DELE também, e
# /content existe. Testar Colab primeiro fazia o Kaggle se identificar como
# Colab e cair no files.upload() — um seletor de arquivo que ninguém pode
# clicar numa execução em background. O notebook ficava parado, com Output 0 B.
# /kaggle/working só existe no Kaggle, então ele decide primeiro.
EM_KAGGLE = os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
EM_COLAB = not EM_KAGGLE and ("google.colab" in sys.modules or os.path.exists("/content"))
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else "/content" if EM_COLAB else ".")
print("ambiente:", "Kaggle" if EM_KAGGLE else "Colab" if EM_COLAB else "local", "| base:", BASE)

URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
BRANCH = "poc/tres-coordenadas"   # variantes do ST-GCN
REPO = BASE / "libras-livre-ai-glasses-brasil"
TREINO = REPO / "computer-vision-model" / "treino"

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    return subprocess.run(cmd, capture_output=True, text=True)

# Esta célula deixa o código SEMPRE atual, em três situações diferentes:
#   1. não há clone            -> clona a branch certa
#   2. há clone, branch errada -> descarta e reclona (foi o que aconteceu quando
#                                 o clone veio da default e não tinha treino/)
#   3. há clone, branch certa  -> ATUALIZA. Sem isso, reabrir o notebook num
#                                 runtime que ainda vive reaproveita código velho
#                                 e as correções recém-publicadas não chegam.
if REPO.exists() and (git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip() != BRANCH
                      or not TREINO.is_dir()):
    print("clone existente está na branch errada ou incompleto — refazendo")
    shutil.rmtree(REPO)

if REPO.exists():
    antes = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    git("fetch", "--depth", "1", "origin", BRANCH, repo=REPO)
    git("reset", "--hard", f"origin/{BRANCH}", repo=REPO)
    depois = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    print(f"repo atualizado: {antes} -> {depois}" if antes != depois else
          f"repo já estava atual ({depois})")
else:
    # --branch no clone: sem isso vem a default (main), que não tem treino/.
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL,
                            str(REPO)], capture_output=True, text=True)
    if clone.returncode:
        dica = ("\n\nNO KAGGLE: internet vem DESLIGADA por padrão. Abra o painel da\n"
                "direita -> Notebook options -> Internet: On (exige telefone\n"
                "verificado na conta). Sem isso o clone não tem como funcionar."
                if EM_KAGGLE else "")
        raise SystemExit(f"git clone falhou:\n{clone.stderr.strip()}{dica}")

assert TREINO.is_dir(), f"{TREINO} não existe mesmo após o clone"
print("branch:", git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip())
print("último commit:", git("log", "-1", "--pretty=%h %s", repo=REPO).stdout.strip())
print("código em:", TREINO)


In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative a GPU antes de continuar; não executar treino pesado em CPU.")
print("GPU:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scipy"], check=True)

## 2. Landmarks MINDS

In [ ]:
import json
import runpy

# Só MINDS: esta PoC compara representações no LOSO, sem pré-treino V-LIBRASIL.
# Deixe vazio para descoberta automática. Se houver vários candidatos, informe
# o caminho completo da pasta landmarks OU do landmarks-minds.tar.gz desejado.
ORIGEM_MINDS = ""
DESTINO = (REPO / "computer-vision-model" / "PoC" / "data").resolve()

# run_path usa o helper do clone atualizado, sem reaproveitar um import antigo.
entrada = runpy.run_path(str(TREINO / "entrada_poc.py"))
if EM_COLAB and not ORIGEM_MINDS:
    from google.colab import files
    print("Selecione o pacote privado: landmarks-minds.tar.gz")
    enviados = files.upload()
    if set(enviados) != {entrada["PACOTE"]}:
        raise ValueError("Envie exatamente landmarks-minds.tar.gz.")
    origem = pathlib.Path(entrada["PACOTE"]).resolve()
    del enviados
elif EM_KAGGLE:
    origem = entrada["localizar_minds"](pathlib.Path("/kaggle/input"), ORIGEM_MINDS)
else:
    # Local/Colab com caminho explícito aceita arquivo ou pasta, sem upload.
    origem = entrada["localizar_minds"](
        pathlib.Path.home(), ORIGEM_MINDS or pathlib.Path.home() / entrada["PACOTE"])

print("Entrada MINDS:", origem)
entrada["preparar_minds"](origem, DESTINO / "landmarks")
print("Dados prontos em:", DESTINO / "landmarks")

## 3. Preparação e orçamento de tempo

In [ ]:
import tarfile  # usado na célula final; sem isto ela morre de NameError DEPOIS dos treinos
import time
from datetime import datetime
from uuid import uuid4

TREINO = TREINO.resolve()
INICIO = time.time()
LIMITE_HORAS = 8.0        # margem sob o limite de sessão do Kaggle
MIN_POR_EXEC = 60         # medido: 55 min; arredondado para cima

EXP = (BASE.resolve() / "experimentos-privados" /
       ("gcn-" + datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]))
EXP.mkdir(parents=True, exist_ok=False)
print("resultados em:", EXP)

SEMENTE = "20260916"      # a mesma da sessão anterior: mantém tudo pareado
BASE_ARGS = [
    "--arquitetura", "gcn", "--fontes", "minds", "--dispositivo", "cuda",
    "--epocas", "120", "--lr", "1e-3", "--batch", "64", "--agendador", "cosseno",
    "--folds", "0", "--semente", SEMENTE,
]

# Todas partem de --ossos, que é o que já venceu. Cada uma muda UMA coisa.
VARIANTES = {
    "G-ossos":        ["--ossos"],
    "G-ossos-z":      ["--ossos", "--com-z", "--z-recentrado"],
    "G-ossos-mov":    ["--ossos", "--movimento"],
    "G-ossos-z-mov":  ["--ossos", "--com-z", "--z-recentrado", "--movimento"],
    "G-ossos-adapt":  ["--ossos", "--adjacencia-adaptativa"],
    "G-ossos-k5":     ["--ossos", "--kernel-temporal", "5"],
}

def cabe(minutos=MIN_POR_EXEC):
    """Só começa o que dá para terminar — ser morto no meio perde a sessão inteira."""
    return (time.time() - INICIO) / 3600 + minutos / 60 <= LIMITE_HORAS

def rodar(nome, extra, args=None):
    if (EXP / nome / "relatorio.md").is_file():
        print(f"[pular] {nome} já existe"); return True
    if not cabe():
        print(f"[ORÇAMENTO] {nome} não cabe em {LIMITE_HORAS}h — pulando"); return False
    t0 = time.time()
    print("\n" + "=" * 72 + f"\n### {nome}  ({(time.time()-INICIO)/3600:.1f}h decorridas)\n" + "=" * 72,
          flush=True)
    subprocess.run([sys.executable, "treinar.py", *(args or BASE_ARGS), *extra,
                    "--saida", str(EXP / nome)], cwd=TREINO, check=True)
    print(f"[ok] {nome} em {(time.time()-t0)/60:.0f} min", flush=True)
    return True

subprocess.run([sys.executable, "selftest.py"], cwd=TREINO, check=True)
print("selftest OK — pode treinar")


## 4. As seis variantes

In [ ]:
for nome, extra in VARIANTES.items():
    rodar(nome, extra)
print(f"\nvariantes concluídas em {(time.time()-INICIO)/3600:.1f}h")


## 5. Confirmação do vencedor com outras sementes

In [ ]:
import re

def ler(nome):
    t = (EXP / nome / "relatorio.md").read_text(encoding="utf-8")
    folds = {k: float(v) for k, v in re.findall(r"^\| (M\d+) \| ([\d.]+)%", t, re.M)}
    return folds, float(re.search(r"média = ([\d.]+)%", t).group(1))

feitas = [n for n in VARIANTES if (EXP / n / "relatorio.md").is_file()]
candidatos = [n for n in feitas if n != "G-ossos"]
if not candidatos:
    print("nenhuma variante concluída — nada a confirmar")
else:
    melhor = max(candidatos, key=lambda n: ler(n)[1])
    print(f"melhor da sessão: {melhor} ({ler(melhor)[1]:.1f}%) "
          f"vs controle G-ossos ({ler('G-ossos')[1]:.1f}%)")

    # POR QUE O CONTROLE VAI JUNTO. Escolher o máximo entre 5 candidatos infla o
    # número — parte do ganho é sorte. Rerodar só o vencedor testa sensibilidade à
    # semente, mas NÃO separa "a variante é melhor" de "esta semente é boa": sem o
    # controle na mesma semente não há com o que comparar. Por isso vão os dois.
    # Isso ainda não é estimativa livre de viés (a escolha usou as mesmas pessoas
    # de teste); para isso seria preciso avaliação externa ou validação aninhada.
    for s in ("20260917",):
        args = [a if a != SEMENTE else s for a in BASE_ARGS]
        for nome in ("G-ossos", melhor):
            if not rodar(f"{nome}-sem{s[-2:]}", VARIANTES[nome], args):
                break


## 6. Tabela comparativa

In [ ]:
import re

def ler(nome):
    t = (EXP / nome / "relatorio.md").read_text(encoding="utf-8")
    folds = {k: float(v) for k, v in re.findall(r"^\| (M\d+) \| ([\d.]+)%", t, re.M)}
    return folds, float(re.search(r"média = ([\d.]+)%", t).group(1))

nomes = [n for n in VARIANTES if (EXP / n / "relatorio.md").is_file()]
res = {n: ler(n) for n in nomes}
ctrl = res["G-ossos"][0]
pessoas = sorted(ctrl)

print(f"{'fold':<7}" + "".join(f"{n:>20}" for n in nomes))
print("-" * (7 + 20 * len(nomes)))
for p in pessoas:
    linha = f"{p:<7}"
    for n in nomes:
        a = res[n][0][p]
        linha += f"{a:>11.1f}%" + ("        " if n == "G-ossos" else f"{a-ctrl[p]:>+7.1f} ")
    print(linha)
print("-" * (7 + 20 * len(nomes)))
linha = f"{'MÉDIA':<7}"
for n in nomes:
    m = res[n][1]
    linha += f"{m:>11.1f}%" + ("        " if n == "G-ossos" else f"{m-res['G-ossos'][1]:>+7.1f} ")
print(linha)
for n in nomes:
    if n == "G-ossos":
        continue
    d = [res[n][0][p] - ctrl[p] for p in pessoas]
    print(f"  {n}: {sum(x>0 for x in d)}/8 folds acima, {sum(x<0 for x in d)} abaixo")

print("\nCONFIRMAÇÃO (mesma variante, sementes diferentes):")
for n in sorted(p.name for p in EXP.iterdir() if "-sem" in p.name):
    if (EXP / n / "relatorio.md").is_file():
        print(f"  {n}: {ler(n)[1]:.1f}%")

print("""
COMO LER. Referência da sessão anterior: ST-GCN+ossos 91,0% e ResNet-18 93,0%.
Diferença de média abaixo de ~1,5 pp é empate; olhe a contagem de folds.
A confirmação é o que separa ganho real de sorte na escolha do máximo — se a
variante vencedora cair perto do controle com outra semente, o ganho era ruído.
""")


## 7. Artefatos

In [ ]:
for rel in sorted(EXP.rglob("relatorio.md")):
    print("=" * 70, "\n", rel.relative_to(EXP))
    print(rel.read_text(encoding="utf-8")[:1500])

artefatos = sorted(p for p in EXP.rglob("*") if p.is_file())
if not artefatos:
    raise RuntimeError("Nenhum artefato para entregar.")
for artefato in artefatos:
    print(artefato.relative_to(EXP), "|", artefato.stat().st_size, "bytes")

# Arquivo fora de EXP para não incluir a si próprio; preserva todos os artefatos.
arquivo = EXP.parent / f"{EXP.name}.tar.gz"
with tarfile.open(arquivo, "w:gz") as tar:
    tar.add(EXP, arcname=EXP.name)
print("\nArquivo completo PRIVADO:", arquivo)

if EM_KAGGLE:
    # Nada a baixar aqui: o download do navegador nem existiria numa execução em
    # background. O .tar.gz está em /kaggle/working e sai como output da versão —
    # aba "Output" da versão, ou "Download all".
    print("Kaggle: pegue o arquivo no Output desta versão (não precisa baixar agora).")
elif EM_COLAB:
    from google.colab import files
    files.download(str(arquivo))
else:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(arquivo, pathlib.Path.cwd())))
